# Mixed Traffic FCT Plots

This notebook plots the mixed traffic experiment from `results/FCT_mixed`. 
Fat-tree is included in the network list even if its logs are not generated yet; missing logs are skipped and will appear once the experiment is run.


In [ ]:
import os
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
from fct_parallel import (
    build_fct_data_from_network_results,
    collect_all_metric_network_results,
    compute_fct_data_average,
    run_all_metrics as run_all_fct_metrics,
    run_fct_plot as run_single_fct_plot,
)
from plot_utils import (
    DEFAULT_DPI,
    FIG_DIR,
    FONT_SIZE,
    TEXT_WIDTH,
    add_flow_size_regions,
    adjust_subplot_widths,
    metric_yticks,
    metric_norm_yticks,
    plot_variable_lines,
    save_and_trim,
    style_axis,
)

try:
    from IPython.display import display
except ImportError:
    display = print


In [ ]:
XTICKS_MIXED = [10**2, 10**3, 10**4, 10**5, 10**6, 10**7, 10**8, 10**9]
YTICKS_AVG = [10**-3, 10**-2, 10**-1, 10**0, 10**1, 10**2, 10**3, 10**4]
YTICKS_P99 = [10**-3, 10**-2, 10**-1, 10**0, 10**1, 10**2, 10**3, 10**4]
YTICKS_HD_AVG = YTICKS_AVG
YTICKS_HD_P99 = YTICKS_P99
NORM_YTICKS_HD_AVG = [0.5, 1, 1.5, 2.0, 2.5, 3.0]
NORM_YTICKS_HD_P99 = [0.5, 1, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0]
LONG_FIG_RATIO = 0.225
CUTOFF = 60_000_000
DEFAULT_ALPHA = 0.18
VERBOSE = False
TRIM_WHITE_THRESHOLD = 250
TRIM_VERTICAL_PAD = 5

EXP_NAME = "FCT_mixed"
FILE_TEMPLATE = "../../results/FCT_mixed/log_{network}_18Ngroup_mixed_{variable}pload_seed={seed}.txt"
ALTERNATE_FILE_TEMPLATES = {
    "clos_prio_128q": [
        "../../results/FCT_mixed/log_clos_prio_18Ngroup_mixed_{variable}pload_128q_seed={seed}.txt",
    ],
}
NETWORKS = ["cbb_108i", "opera_ecmp", "clos_prio_128q"]
LABELS = ["CBB-Net", "Opera", "3:1 Fat-tree"]
VARIABLE_NAME = "load"
VARIABLE_VALUES = ["5.00", "10.00", "15.00", "20.00", "25.00", "30.00"]
MAX_LOAD_BY_NETWORK = {
    "opera_ecmp": 25.0,
    "clos_prio_128q": 20.0,
}
SEEDS = ["1", "2", "3", "4", "5"]
METRICS = ["avg", "p99"]
FCT_MAX_WORKERS = 3
XTICKS = XTICKS_MIXED
XLIM = (XTICKS_MIXED[0], 2 * 10**9)
_FCT_METRIC_CACHE = {}
_MISSING_FCT_FILES = set()


In [ ]:
# Common plotting helpers are imported from plot_utils.py.


In [ ]:
def plot_fct_results(
    fct_results_df,
    variable_name,
    variable_values,
    labels,
    fct_metric,
    fig_name,
    xticks,
    xlim,
):
    """Plot Microsoft FCT results for one workload/metric and save the figure."""
    num_networks = len(fct_results_df)
    if len(labels) != num_networks:
        raise ValueError("labels must have one entry for each network result")

    fig_width = TEXT_WIDTH
    fig_height = LONG_FIG_RATIO * fig_width
    first_yticks = metric_yticks(fct_metric, YTICKS_HD_AVG, YTICKS_HD_P99)
    # normalized_yticks = [0.6, 0.8, 1, 1.2, 1.4, 1.6, 1.8, 2.0, 2.2, 2.4, 2.6]
    # normalized_yticks = [0.6, 1, 1.4, 1.8, 2.2, 2.6]
    # normalized_yticks = [0.5, 1, 1.5, 2.0, 2.5, 3.0]
    normalized_yticks = metric_norm_yticks(fct_metric, NORM_YTICKS_HD_AVG, NORM_YTICKS_HD_P99)
    legend_handles = []
    legend_labels = []

    fig, axes = plt.subplots(
        1,
        num_networks,
        figsize=(fig_width, fig_height),
        sharey=False,
        dpi=DEFAULT_DPI,
    )
    axes = [axes] if num_networks == 1 else list(axes)

    for idx, (_, df) in enumerate(fct_results_df.items()):
        ax = axes[idx]
        add_flow_size_regions(ax, xlim, legend_handles, legend_labels)
        plot_variable_lines(ax, df, variable_name, variable_values, legend_handles, legend_labels)
        style_axis(ax, idx, labels[idx], fct_metric, xticks, xlim, first_yticks, normalized_yticks)

    fig.legend(
        handles=legend_handles,
        labels=legend_labels,
        handlelength=1.25,
        markerscale=1.0,
        loc="center right",
        fontsize=FONT_SIZE - 2,
        frameon=True,
    )
    plt.tight_layout(rect=[0, 0, 0.92, 1])
    adjust_subplot_widths(axes)

    Path(FIG_DIR).mkdir(parents=True, exist_ok=True)
    fig_path = Path(FIG_DIR) / f"sec6_{fig_name}.png"
    save_and_trim(fig_path, dpi=DEFAULT_DPI, threshold=TRIM_WHITE_THRESHOLD, pad=TRIM_VERTICAL_PAD, verbose=VERBOSE)
    plt.show()


In [ ]:
def run_fct_plot(fct_metric, display_network="opera_ecmp"):
    """Compute averaged FCT data and plot one mixed-traffic metric."""
    return run_single_fct_plot(
        fct_metric,
        exp_name=EXP_NAME,
        file_template=FILE_TEMPLATE,
        networks=NETWORKS,
        variable_name=VARIABLE_NAME,
        variable_values=VARIABLE_VALUES,
        seeds=SEEDS,
        labels=LABELS,
        xticks=XTICKS,
        xlim=XLIM,
        plot_fct_results_func=plot_fct_results,
        display_func=display,
        display_network=display_network,
        alternate_file_templates=ALTERNATE_FILE_TEMPLATES,
        max_load_by_network=MAX_LOAD_BY_NETWORK,
        max_workers=FCT_MAX_WORKERS,
        verbose=VERBOSE,
    )


def run_all_metrics(display_network="opera_ecmp"):
    """Run average and p99 mixed-traffic plots after reading logs once."""
    return run_all_fct_metrics(
        exp_name=EXP_NAME,
        file_template=FILE_TEMPLATE,
        networks=NETWORKS,
        metrics=METRICS,
        variable_name=VARIABLE_NAME,
        variable_values=VARIABLE_VALUES,
        seeds=SEEDS,
        labels=LABELS,
        xticks=XTICKS,
        xlim=XLIM,
        plot_fct_results_func=plot_fct_results,
        display_func=display,
        display_network=display_network,
        alternate_file_templates=ALTERNATE_FILE_TEMPLATES,
        max_load_by_network=MAX_LOAD_BY_NETWORK,
        max_workers=FCT_MAX_WORKERS,
        verbose=VERBOSE,
    )


## Run Mixed Traffic Plots

Run both average and p99 FCT plots for mixed traffic.


In [ ]:
mixed_results = run_all_metrics()
